# Notebook 02 - Logistic Regression Benchmark (RQ2)

**RQ2:** Which attribution model best reflects the actual conversion behavior observed in the data?


In [1]:
import sys
from itertools import combinations
from pathlib import Path

for candidate in [Path.cwd().resolve(), Path.cwd().resolve() / "notebooks", Path.cwd().resolve() / "analysis_python" / "notebooks", Path.cwd().resolve().parent / "notebooks"]:
    if (candidate / "notebook_header.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find notebook_header.py")

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.special import rel_entr
from scipy.stats import chi2_contingency, spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from notebook_header import OUTPUT_DIR as BASE_OUTPUT_DIR, RANDOM_SEED, load_encoded, load_journeys

OUTPUT_DIR = BASE_OUTPUT_DIR / "rq2_logit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
np.random.seed(RANDOM_SEED)

## 1. Load Cleaned Regression Inputs

`data_encoded.csv` already contains one-hot channel columns. `data_journeys.csv` contains user-level conversion and journey length.

In [2]:
df_enc = load_encoded()
df_jr = load_journeys()

channel_cols = [col for col in df_enc.columns if col.startswith("Channel_")]

assert len(df_enc) == 10_000
assert len(df_jr) == 2_847
assert len(channel_cols) == 6
assert int(df_jr["Converted"].sum()) == 2_381

print(f"Encoded touchpoints: {df_enc.shape}")
print(f"Journeys: {df_jr.shape}")
print(f"Channel dummy columns: {channel_cols}")

Encoded touchpoints: (10000, 10)
Journeys: (2847, 9)
Channel dummy columns: ['Channel_Direct Traffic', 'Channel_Display Ads', 'Channel_Email', 'Channel_Referral', 'Channel_Search Ads', 'Channel_Social Media']


## 2. Build Modeling Tables

Two levels are tested:

- **Row-level channel model:** predicts row-level `Conversion` from the channel of the touchpoint. This tests whether individual channel rows have strong direct signal.
- **User-level models:** predict whether a user converted. These models can be affected by journey length, so a length-only baseline and an adjusted channel-plus-length model are included.

In [3]:
row_y = (df_enc["Conversion"] == "Yes")

df_user = (
    df_enc.groupby("User ID")[channel_cols]
    .max()
    .reset_index()
    .merge(df_jr[["User ID", "Converted", "N_Touchpoints"]], on="User ID", how="inner")
)

assert len(df_user) == 2_847
assert int(df_user["Converted"].sum()) == 2_381

contingency = pd.crosstab(df_enc[channel_cols].idxmax(axis=1).str.replace("Channel_", "", regex=False), df_enc["Conversion"])
chi2, chi_p, chi_dof, _ = chi2_contingency(contingency)

print(f"Channel x row Conversion chi-square: chi2={chi2:.4f}, p={chi_p:.4f}, dof={chi_dof}")
print(f"User-level converted rate: {df_user['Converted'].mean()*100:.2f}%")
print(f"Mean journey length: {df_user['N_Touchpoints'].mean():.2f}")

Channel x row Conversion chi-square: chi2=1.9220, p=0.8598, dof=5
User-level converted rate: 83.63%
Mean journey length: 3.51


## 3. Fit Logistic Models

In [4]:
def fit_logit(model_name, X, y):
    """Fit a logit model and return the fitted model plus paper-ready metrics."""
    X_model = X.astype(float)
    y_model = y.astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X_model, y_model, test_size=0.30, random_state=RANDOM_SEED, stratify=y_model
    )
    fitted = sm.Logit(y_train, sm.add_constant(X_train, has_constant="add")).fit(disp=0, maxiter=200)
    y_score = fitted.predict(sm.add_constant(X_test, has_constant="add"))
    return fitted, {
        "model": model_name,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "positive_rate": float(y_model.mean()),
        "pseudo_r2_mcfadden": float(fitted.prsquared),
        "auc_test": float(roc_auc_score(y_test, y_score)),
    }

model_objects = {}
metrics_rows = []

# Drop the first one-hot column only for the row-level model to avoid the dummy-variable trap.
model_objects["row_channel"] , metrics = fit_logit("row_channel", df_enc[channel_cols[1:]], row_y)
metrics_rows.append(metrics)

model_objects["user_any_channel"], metrics = fit_logit("user_any_channel", df_user[channel_cols], df_user["Converted"])
metrics_rows.append(metrics)

model_objects["journey_length_only"], metrics = fit_logit("journey_length_only", df_user[["N_Touchpoints"]], df_user["Converted"])
metrics_rows.append(metrics)

model_objects["channel_plus_length"], metrics = fit_logit("channel_plus_length", df_user[channel_cols + ["N_Touchpoints"]], df_user["Converted"])
metrics_rows.append(metrics)

model_metrics = pd.DataFrame(metrics_rows).round(4)
model_metrics

,model,n_train,n_test,positive_rate,pseudo_r2_mcfadden,auc_test
0,row_channel,7000,3000,0.4944,0.0002,0.4902
1,user_any_channel,1992,855,0.8363,0.1720,0.7215
2,journey_length_only,1992,855,0.8363,0.1883,0.7549
3,channel_plus_length,1992,855,0.8363,0.1960,0.7536


## 4. Adjusted Channel Coefficients

The `channel_plus_length` model is used as the main benchmark for RQ2 because it controls for the fact that longer journeys are mechanically more likely to contain at least one conversion-positive touchpoint.

In [5]:
adjusted_model = model_objects["channel_plus_length"]
conf = adjusted_model.conf_int()
conf.columns = ["ci_low", "ci_high"]

adjusted_coefficients = pd.DataFrame({
    "coef": adjusted_model.params,
    "odds_ratio": np.exp(adjusted_model.params),
    "or_ci_low": np.exp(conf["ci_low"]),
    "or_ci_high": np.exp(conf["ci_high"]),
    "p_value": adjusted_model.pvalues,
}).round(4)

channel_beta = adjusted_model.params[channel_cols].copy()
channel_beta.index = channel_beta.index.str.replace("Channel_", "", regex=False)
channel_score = np.exp(channel_beta - channel_beta.max())
logit_adjusted_share = (channel_score / channel_score.sum() * 100).round(2).sort_values(ascending=False)

adjusted_coefficients

,coef,odds_ratio,or_ci_low,or_ci_high,p_value
const,-0.9742,0.3775,0.2725,0.5229,0.0000
Channel_Direct Traffic,0.3047,1.3562,0.9326,1.9722,0.1108
Channel_Display Ads,0.6464,1.9087,1.2927,2.8183,0.0011
Channel_Email,0.4375,1.5488,1.0486,2.2875,0.0279
Channel_Referral,0.2950,1.3431,0.9255,1.9493,0.1206
Channel_Search Ads,0.0957,1.1004,0.7571,1.5993,0.6160
Channel_Social Media,0.3531,1.4235,0.9793,2.0691,0.0642
N_Touchpoints,0.6478,1.9114,1.5432,2.3676,0.0000


## 5. Compare Heuristic Attribution Models with Adjusted Logistic Benchmark

In [6]:
attribution = pd.read_csv(OUTPUT_DIR.parent / "rq1_basic" / "01_attribution_share.csv", index_col=0)
attribution = attribution.reindex(logit_adjusted_share.index)

spearman_rows = []
for col in ["first_touch_pct", "last_touch_pct", "linear_pct"]:
    rho, p_value = spearmanr(attribution[col], logit_adjusted_share)
    spearman_rows.append({
        "heuristic_model": col,
        "benchmark": "logit_adjusted_channel_share",
        "spearman_rho": round(float(rho), 4),
        "p_value": round(float(p_value), 4),
    })

spearman_vs_logit = pd.DataFrame(spearman_rows)

distributions = attribution.copy()
distributions["logit_adjusted_pct"] = logit_adjusted_share

def kl_divergence_pct(p, q):
    p = np.asarray(p, dtype=float) / 100
    q = np.asarray(q, dtype=float) / 100
    eps = 1e-12
    p = np.clip(p, eps, None)
    q = np.clip(q, eps, None)
    p = p / p.sum()
    q = q / q.sum()
    return float(np.sum(rel_entr(p, q)))

kl_rows = []
for p_col, q_col in combinations(distributions.columns, 2):
    kl_rows.append({"P": p_col, "Q": q_col, "KL(P||Q)": round(kl_divergence_pct(distributions[p_col], distributions[q_col]), 6)})

kl_divergence = pd.DataFrame(kl_rows)

print("Adjusted logistic channel share (%):")
print(logit_adjusted_share)
print("\nSpearman vs adjusted logistic benchmark:")
print(spearman_vs_logit)
print("\nKL divergence:")
print(kl_divergence)

Adjusted logistic channel share (%):
Display Ads       21.99
Email             17.84
Social Media      16.40
Direct Traffic    15.62
Referral          15.47
Search Ads        12.68
dtype: float64

Spearman vs adjusted logistic benchmark:
   heuristic_model                     benchmark  spearman_rho  p_value
0  first_touch_pct  logit_adjusted_channel_share        0.4857   0.3287
1   last_touch_pct  logit_adjusted_channel_share        0.0857   0.8717
2       linear_pct  logit_adjusted_channel_share        0.3143   0.5441

KL divergence:
                 P                   Q  KL(P||Q)
0  first_touch_pct      last_touch_pct    0.0013
1  first_touch_pct          linear_pct    0.0005
2  first_touch_pct  logit_adjusted_pct    0.0100
3   last_touch_pct          linear_pct    0.0005
4   last_touch_pct  logit_adjusted_pct    0.0143
5       linear_pct  logit_adjusted_pct    0.0119


## 6. Save Outputs

In [7]:
chi_square_summary = pd.DataFrame([{
    "test": "channel_by_row_conversion",
    "chi_square": round(float(chi2), 6),
    "p_value": round(float(chi_p), 6),
    "degrees_of_freedom": int(chi_dof),
}])

rq2_interpretation = pd.DataFrame([
    {"finding": "Row-level channel signal is weak", "evidence": "row_channel AUC is near 0.50 and chi-square p-value is high"},
    {"finding": "User-level prediction improves substantially", "evidence": "any-channel exposure and journey length both predict user-level conversion"},
    {"finding": "Journey length is a major confounder", "evidence": "journey_length_only performs at least as well as user_any_channel"},
    {"finding": "Adjusted logistic benchmark is conservative", "evidence": "channel_plus_length is used for attribution-model comparison"},
])

model_metrics.to_csv(OUTPUT_DIR / "02_model_metrics.csv", index=False)
adjusted_coefficients.to_csv(OUTPUT_DIR / "02_adjusted_coefficients.csv")
logit_adjusted_share.to_frame("logit_adjusted_share_pct").to_csv(OUTPUT_DIR / "02_adjusted_channel_share.csv")
spearman_vs_logit.to_csv(OUTPUT_DIR / "02_spearman_vs_logit.csv", index=False)
kl_divergence.to_csv(OUTPUT_DIR / "02_kl_divergence.csv", index=False)
chi_square_summary.to_csv(OUTPUT_DIR / "02_channel_conversion_chi_square.csv", index=False)
rq2_interpretation.to_csv(OUTPUT_DIR / "02_interpretation_notes.csv", index=False)

print("Saved RQ2 outputs to outputs/rq2_logit/.")

Saved RQ2 outputs to outputs/rq2_logit/.

## 7. RQ2 conclusion

RQ2 shows that the link between channel and conversion is weak when channel is tested on its own. At the row level, the channel-only logistic model produces an AUC of 0.4902 and a McFadden Pseudo-R² of only 0.0002, which means it has almost no useful classification power. The chi-square test between Channel and row-level Conversion is also not statistically significant (χ² = 1.9220, p = 0.8598). This fits the RQ1 pattern: conversion rates across channels are close to one another, so channel identity alone does not explain much of the outcome.

At the user level, the models perform better, but the reason matters. It is worth noting first that the user-level dataset has a positive rate of 83.63%, which provides useful context for interpreting the AUC values. The user-level model using channel variables reaches AUC = 0.7215, while the journey-length-only model reaches AUC = 0.7549, even though it uses no channel information at all. In this dataset, knowing how many touchpoints a user had is more informative than knowing which channels appeared in the journey. This is an important limitation: longer journeys are mechanically more likely to include at least one conversion-positive touchpoint, which means a channel-only model risks overstating how much individual channels actually contribute.

Given this, the adjusted channel_plus_length model is used as the main benchmark. It reaches AUC = 0.7536 and McFadden Pseudo-R² = 0.1960. Its AUC is slightly lower than the journey-length-only model (AUC = 0.7549), which means the channel variables add interpretive value but do not improve predictive performance. Consistent with the earlier finding, N_Touchpoints is strongly significant in this model (odds ratio = 1.9114, p < 0.001), showing that journey length remains strongly associated with conversion even after controlling for channel. The adjusted channel ranking is Display Ads (21.99%), Email (17.84%), Social Media (16.40%), Direct Traffic (15.62%), Referral (15.47%), and Search Ads (12.68%). Among the channels, Display Ads is statistically significant (p = 0.0011) and Email is also significant (p = 0.0279), while Social Media reaches only marginal significance (p = 0.0642).

Compared with this benchmark, First-Touch has the highest Spearman correlation (rho = 0.4857, p = 0.3287), followed by Linear attribution (rho = 0.3143, p = 0.5441) and Last-Touch (rho = 0.0857, p = 0.8717). None of these correlations is statistically significant, so First-Touch is the closest of the three heuristics to the logistic benchmark, but that is a relative comparison, not an endorsement of it as a strong model.

Taken together, none of the three heuristic attribution models fully reflects the conversion behavior estimated by the regression benchmark. First-Touch is the closest match among them, but the more consequential finding is that journey length explains more than channel choice at the user level. The regression model is useful as a comparison point, though it should not be treated as causal evidence that specific channels directly drove conversions.
